# GeoNUSAF Baseline — U-Net (ResNet-18, ImageNet), regularized

**Kathmandu Valley satellite land-use segmentation — 6 classes + ignore**

This replaces the ResNet-50 U-Net. Two things changed and both matter for the paper.

## 1. Capacity
| | encoder | decoder | total |
|---|---|---|---|
| old: `resnet50` + (256,128,64,32,16) | 23.5 M | 9.0 M | **~32.5 M** |
| new: `resnet18` + (128,64,32,16,8) | 11.2 M | 1.3 M | **~12.5 M** |

12.5 M puts U-Net alongside **UNetFormer (11.7 M)**, which uses the *same ResNet-18 encoder*. That
is not a coincidence to hide — it is the point. With the encoder held constant, U-Net vs UNetFormer
becomes a controlled decoder comparison (plain CNN decoder vs global-local transformer decoder),
which is a far stronger claim than "we also ran a U-Net". Section 25 has the sentence to put in the paper.

**Keep the ResNet-50 results.** They are already trained and cost nothing to report. Two U-Net rows
(32.5 M and 12.5 M) give you a capacity ablation inside the CNN family that independently supports
dropping SegFormer-B2.

## 2. Regularization — the part that was actually broken
The old U-Net run had **no EMA, no label smoothing, no dropout, and weight decay 1e-4**, while
`segformer-b0-new.ipynb` had EMA 0.999, label smoothing 0.05, dropout 0.1, drop-path 0.1 and
weight decay 0.01. Train loss fell to ~0.05 while val loss bottomed at epoch 8-10 and climbed to ~2.8.
Some of that is capacity, but a large part is simply that **U-Net was regularized less than the model
it was being compared against** — which a reviewer can read as a crippled baseline.

Section 2 defines a `# ---- REGULARIZATION` block. **That block is now protocol.** Copy it verbatim
into all five notebooks. Section 25 lists exactly what to change in each.

## What is fixed by protocol (do not change per-model)
| Item | Value |
|---|---|
| Split | 3-fold CV over **408 base tiles** (flip variants travel as one group) |
| Split modes | `block` (export-sequence proxy) **and** `random` (leakage-gap control) |
| Loss | effective-number-weighted CE (auto-beta, capped) + Lovasz-Softmax, lambda=0.75 |
| Label smoothing | 0.05 |
| Optimizer | AdamW, differential LR (encoder = decoder/10), **no decay on norm/bias** |
| Weight decay | 0.01 |
| Schedule | 500-step linear warmup, cosine over a **120-epoch horizon**, max 200 epochs |
| Dropout | 0.10 (bottleneck + pre-head for CNNs; classifier + drop-path for transformers) |
| Weight averaging | EMA 0.999 — **validation and checkpointing use EMA weights** |
| Early stopping | val mIoU, patience 25, `MIN_DELTA` 0.002 |
| Stability | `clip_grad_norm_(1.0)` after `scaler.unscale_`, grad-norm logged |
| Augmentation | rot90 (k in 0..3) + mild photometric jitter, on-the-fly. Baked-in flips never repeated. |
| Seed | single seed = 42 |
| Ignore | index 255, excluded from loss and all metrics |

## What is model-specific here
- `smp.Unet`, encoder `resnet18` (ImageNet), decoder channels (128,64,32,16,8)
- LR: decoder 1e-4 / encoder 1e-5

## An honest note on "overfitting-free"
There is no such thing at 408 independent scenes. What this notebook does is remove the *avoidable*
overfitting — the asymmetry against SegFormer, the missing weight averaging, the cosine that never
completed, the 2.6x capacity excess — and make the remaining gap an honest property of the dataset
rather than an artifact of the setup. Expect the train/val gap to shrink and the best epoch to move
later, not to vanish. The gap itself is now logged every epoch (`gen_gap`) so you can report it.

## Session strategy (12 h Kaggle cap)
Each **fold is one session**. Full state (incl. EMA) is pushed to Hugging Face as `last.pt`; on restart
the notebook resumes automatically. Set `FOLD` and `SPLIT_MODE`, run top to bottom, repeat.

## Prerequisites
- **Internet ON** (Settings -> Internet). Needed for pip, ImageNet weights, W&B, HF.
- Kaggle Secrets: `HF_TOKEN` (write scope) and `WANDB_API_KEY`.
- GPU: T4 x1.

## 1. Environment

`segmentation_models_pytorch` and `ptflops` are not preinstalled on Kaggle. Everything else
(torch, cv2, skimage, sklearn, huggingface_hub, wandb) is. Pinning smp avoids a decoder API
change between 0.3.x and 0.4.x that would silently alter the model you compare against.

The AMP shim exists because `torch.cuda.amp.autocast` / `GradScaler` are deprecated in torch 2.10
(the version now on Kaggle) and emit a warning per call. `amp_autocast` / `make_scaler` resolve to
the new `torch.amp` API when available and fall back otherwise, so the same notebook runs on either.

In [ ]:
!pip install -q segmentation-models-pytorch==0.3.4 ptflops==0.7.3 2>&1 | tail -2

import os, re, gc, json, time, math, copy, random, warnings, itertools
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUs

# ---- AMP compatibility shim (torch >= 2.4 new API, older fallback) ----
try:
    from torch.amp import autocast as _ac, GradScaler as _gs
    def amp_autocast(enabled=True): return _ac("cuda", enabled=enabled)
    def make_scaler(enabled=True):  return _gs("cuda", enabled=enabled)
    _AMP_API = "torch.amp"
except Exception:
    def amp_autocast(enabled=True): return torch.cuda.amp.autocast(enabled=enabled)
    def make_scaler(enabled=True):  return torch.cuda.amp.GradScaler(enabled=enabled)
    _AMP_API = "torch.cuda.amp"

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| amp api", _AMP_API)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration

**Set `FOLD` and `SPLIT_MODE` here, then run the whole notebook.** One fold per session. Those two
fields are the *only* per-session edits.

`SPLIT_MODE`:
- `"block"` — contiguous runs of export-sequence order as a spatial-block proxy (the reported protocol)
- `"random"` — fully random base-tile split (the leakage-gap control)

Both group flip variants, so neither has *duplicate* leakage; the difference isolates *spatial* leakage.

### The `REGULARIZATION` block is shared, not local
Every field under `# ---- REGULARIZATION` must be identical in all five notebooks. If you change one
here, change it everywhere or the comparison is confounded. Field-by-field:

| Field | Value | Why |
|---|---|---|
| `WEIGHT_DECAY` | 0.01 | Matches SegFormer-B0. 1e-4 is far too weak for a randomly-initialized decoder on 816 images. Applied to conv/linear weights only. |
| `NO_DECAY_ON_NORM_BIAS` | True | Decaying BatchNorm gammas and biases shrinks them toward zero and distorts normalization statistics; it regularizes nothing and hurts. Standard practice, rarely done in student code. |
| `LABEL_SMOOTHING` | 0.05 | Your polygon annotations have soft boundaries. Smoothing stops the model from driving logits to infinity on pixels whose label is uncertain by construction. |
| `DROPOUT_BOTTLENECK` | 0.10 | `Dropout2d` on the deepest encoder feature (H/32). Channel-level dropout on the most over-parameterized tensor in the network. |
| `DROPOUT_HEAD` | 0.10 | `Dropout2d` before the 1-conv classifier. Mirrors SegFormer's `classifier_dropout_prob`. |
| `EMA_DECAY` | 0.999 | **The single biggest change.** Validation and checkpointing both use the EMA weights. An EMA at 0.999 over ~51 steps/epoch averages roughly the last 20 epochs of trajectory, which is exactly the noise that makes a small-data run look like it is overfitting when it is really just oscillating. |
| `COS_HORIZON_EPOCHS` | 120 | The old run set cosine over `MAX_EPOCHS=200`, so LR was still at ~35% of peak when early stopping fired at ~100. The schedule never annealed. 120 makes the cosine actually complete inside the realistic run length. |
| `MIN_DELTA` | 0.002 | Patience 25 on *any* improvement means a +0.0001 blip resets the counter and the run drifts 40 epochs past its real peak, memorizing the whole time. `best.pt` still saves on any improvement — only the patience counter uses the threshold. |
| `MAX_EPOCHS` / `PATIENCE` | 200 / 25 | Unchanged, so folds stay comparable to what you already ran. |

`ARCH_VERSION` is stamped into every checkpoint. You hit a resume crash on TC-SegFormer when an HF
repo still held a checkpoint from an older architecture; section 15 now refuses to load a mismatched
`last.pt` and deletes it instead of raising `load_state_dict` errors.

In [ ]:
@dataclass
class CFG:
    # ---- run selection (the ONLY per-session edits) ----
    SPLIT_MODE: str = "block"          # "block" | "random"
    FOLD: int = 0                      # 0 | 1 | 2

    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"
    CACHE_DIR: str = "/kaggle/working/cache512"
    PREBUILT_CACHE: Optional[str] = None    # point at a saved cache dataset to skip the repack
    IMG_SIZE: int = 512
    N_CLASSES: int = 6
    IGNORE_INDEX: int = 255
    PRELOAD_RAM: bool = True           # ~1.3 GB for the whole 512 cache; kills all disk I/O

    # ---- split ----
    N_FOLDS: int = 3
    BLOCK_BUFFER: int = 2              # base tiles dropped from train at each block boundary

    # ---- model (the only model-specific section) ----
    ARCH_VERSION: str = "unet-r18-v1"
    ENCODER: str = "resnet18"
    ENCODER_WEIGHTS: str = "imagenet"
    DECODER_CHANNELS: tuple = (128, 64, 32, 16, 8)
    DECODER_ATTENTION: Optional[str] = None   # "scse" available as an optional ablation
    LR_DECODER: float = 1e-4
    LR_ENCODER: float = 1e-5

    # ---- REGULARIZATION (protocol - identical in ALL five notebooks) ----
    WEIGHT_DECAY: float = 0.01
    NO_DECAY_ON_NORM_BIAS: bool = True
    LABEL_SMOOTHING: float = 0.05
    DROPOUT_BOTTLENECK: float = 0.10
    DROPOUT_HEAD: float = 0.10
    EMA_DECAY: float = 0.999
    EMA_WARMUP_STEPS: int = 2000       # decay ramps in; stops epoch 0 dominating the average
    MAX_EPOCHS: int = 200
    COS_HORIZON_EPOCHS: int = 120      # cosine completes over THIS, not MAX_EPOCHS
    WARMUP_STEPS: int = 500
    PATIENCE: int = 25
    MIN_DELTA: float = 0.002           # patience threshold; best.pt still saves on any improvement
    CLIP_NORM: float = 1.0
    LAMBDA_LOVASZ: float = 0.75
    EN_BETA: float = 0.9999
    WEIGHT_CAP: float = 10.0

    # ---- augmentation (protocol - identical in ALL five notebooks) ----
    AUG_ROT90: bool = True
    AUG_PHOTOMETRIC: bool = True
    AUG_BRIGHTNESS: float = 0.10
    AUG_CONTRAST: float = 0.10

    # ---- optimization ----
    BATCH_SIZE: int = 16
    GRAD_ACCUM: int = 1                # raise to 2 and halve BATCH_SIZE if you OOM

    # ---- runtime ----
    SEED: int = 42
    NUM_WORKERS: int = 4
    AMP: bool = True
    DETERMINISTIC: bool = False        # see section 3 - True costs ~6x on this workload
    RESUME: bool = True

    # ---- persistence ----
    HF_USER: str = "sugam24"           # <-- set to your HF username
    HF_PREFIX: str = "geonusaf-unet-r18"
    PUSH_LAST_EVERY: int = 10          # epochs between full-state pushes
    WANDB_PROJECT: str = "geonusaf-baselines"
    OUT_DIR: str = "/kaggle/working/out"

    MODEL_TAG: str = "UNet-ResNet18"
    CLASS_NAMES: tuple = ("Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural")

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)

RUN_NAME = f"{cfg.HF_PREFIX}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"
print("RUN:", RUN_NAME, "| HF repo:", HF_REPO)
print("arch:", cfg.ARCH_VERSION, "| reg: wd", cfg.WEIGHT_DECAY, "| ls", cfg.LABEL_SMOOTHING,
      "| drop", cfg.DROPOUT_HEAD, "| ema", cfg.EMA_DECAY, "| cos horizon", cfg.COS_HORIZON_EPOCHS)

## 3. Determinism and runtime

**Determinism is off by default, deliberately.** Your D6 attribution found that
`torch.use_deterministic_algorithms(True)` with `cudnn.deterministic=True` forces a sort-based
`indexing_backward_kernel` for bilinear-upsample backward, consuming 63% of all CUDA time — TC-SegFormer
went 2655 ms -> 453 ms per step with it off, plain SegFormer-B0 1967 ms -> 311 ms. The U-Net decoder
upsamples five times per forward, so it pays the same tax. Every baseline you have run so far has been
paying roughly a 6x penalty for a reproducibility guarantee that `warn_only=True` was already breaking.

With `DETERMINISTIC=False` you get `cudnn.benchmark=True`, which autotunes conv algorithms for the fixed
512x512 batch-16 shape. Everything controllable (Python/NumPy/torch RNG, worker seeding, generator, the
per-(epoch, index) augmentation seed) is still pinned, so runs are *statistically* reproducible.

What to write in the paper: seeds are fixed and the data pipeline is deterministic, but cuDNN kernel
selection is not constrained, so results are reproducible to within run-to-run kernel nondeterminism.
That is what the old `warn_only=True` setting actually gave you as well — it just cost 6x to say it.

Set `DETERMINISTIC=True` only if you are chasing a bit-exact reproduction, and expect ~6x the wall clock.

In [ ]:
def set_seed(seed: int, deterministic: bool):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True          # autotune for the fixed 512x512 shape
        torch.use_deterministic_algorithms(False)

def worker_init_fn(worker_id):
    s = cfg.SEED + worker_id
    np.random.seed(s); random.seed(s)

set_seed(cfg.SEED, cfg.DETERMINISTIC)
g = torch.Generator(); g.manual_seed(cfg.SEED)
print(f"seeded: {cfg.SEED} | deterministic={cfg.DETERMINISTIC} "
      f"| cudnn.benchmark={torch.backends.cudnn.benchmark}")
if cfg.DETERMINISTIC:
    print("NOTE: deterministic mode costs roughly 6x wall clock on this workload (see D6).")

## 4. Filename parsing and base-tile grouping

Confirmed convention: `image{N}.png` / `image{N}hf.png` / `image{N}vf.png`, masks mirror with the
`mask` prefix. `N` is ascending **export order with gaps** (mono-class tiles were dropped after export),
and the three variants of a tile are *not* adjacent in filename sort order — so grouping is done by
parsing `N`, never by assuming file adjacency.

The asserts are load-bearing: 1224 files, 408 base tiles, exactly 3 variants each. If any fails, the
dataset version on Kaggle is not the one this protocol was defined against — stop rather than train.

In [ ]:
IMG_DIR  = Path(cfg.DATA_ROOT) / "images"
MASK_DIR = Path(cfg.DATA_ROOT) / "masks"
PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")

records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)
for f in sorted(os.listdir(IMG_DIR)):
    m = PAT.match(f)
    assert m, f"unparsed image filename: {f}"
    base_id, variant = int(m.group(1)), (m.group(2) or "orig")
    mask_name = "mask" + f[len("image"):]
    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"
    records.append((base_id, variant, f, mask_name))

groups: Dict[int, List[tuple]] = {}
for r in records:
    groups.setdefault(r[0], []).append(r)

BASE_IDS = sorted(groups)                      # ascending export order = the spatial proxy axis
assert len(records) == 1224, len(records)
assert len(BASE_IDS) == 408, len(BASE_IDS)
assert all(len(v) == 3 for v in groups.values())

gaps = [b - a for a, b in zip(BASE_IDS, BASE_IDS[1:]) if b - a > 1]
print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")
print(f"{len(gaps)} gaps in export sequence (dropped tiles), total {sum(g-1 for g in gaps)} missing ids")

## 5. Mask colour diagnostic — run this before anything else

Your masks are RGB PNGs; the class table you supplied is in **BGR**. `cv2.imread` returns BGR, so the
keys apply directly — but note that `(128,0,0)` = Residential and `(0,0,128)` = River are exact reverses
of each other. **Getting the channel order wrong silently swaps your largest class with your smallest**,
and every metric in the paper would be wrong in a way no loss curve would reveal.

The orientation guard at the bottom is the check: Residential must dominate River by a wide margin
(D0 measured 42.31% vs 0.57%). If that assert trips, stop and fix the channel order.

In [ ]:
BGR_TO_CLASS = {
    (128, 0, 0): 0,    # Residential Area
    (0, 128, 0): 1,    # Road
    (0, 0, 128): 2,    # River
    (0, 128, 128): 3,  # Forest
    (128, 128, 0): 4,  # Unused Land
    (128, 0, 128): 5,  # Agricultural Area
}

# Pack BGR into a single uint32 key so the mapping is one vectorized lookup, not 6 boolean passes.
def pack_bgr(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

LUT = np.full(1 << 24, 255, dtype=np.uint8)          # 16 MB; default = ignore
for bgr, idx in BGR_TO_CLASS.items():
    LUT[pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0]] = idx

def rgbmask_to_index(bgr_img: np.ndarray) -> np.ndarray:
    return LUT[pack_bgr(bgr_img)]

# ---- diagnostic over a random sample of raw full-res masks ----
rng = np.random.RandomState(cfg.SEED)
sample = [records[i] for i in rng.choice(len(records), 40, replace=False)]
colour_hits, class_px = {}, np.zeros(7, dtype=np.int64)   # 7th bucket = ignore

for _, _, _, mname in sample:
    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)   # BGR
    cols, cnts = np.unique(m.reshape(-1, 3), axis=0, return_counts=True)
    for c, n in zip(cols, cnts):
        colour_hits[tuple(int(x) for x in c)] = colour_hits.get(tuple(int(x) for x in c), 0) + int(n)
    idx = rgbmask_to_index(m)
    b = np.bincount(np.where(idx == 255, 6, idx).ravel(), minlength=7)
    class_px += b

known = set(BGR_TO_CLASS)
print("distinct BGR colours found in sample:")
for c, n in sorted(colour_hits.items(), key=lambda kv: -kv[1]):
    tag = f"-> class {BGR_TO_CLASS[c]} ({cfg.CLASS_NAMES[BGR_TO_CLASS[c]]})" if c in known else "-> IGNORE"
    print(f"  BGR{str(c):<16} {n/sum(colour_hits.values())*100:6.2f}%  {tag}")

tot = class_px.sum()
print("\nclass pixel share (sample):")
for i, nm in enumerate(list(cfg.CLASS_NAMES) + ["<ignore>"]):
    print(f"  {i if i < 6 else 255:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%")

# Orientation guard: Residential must dominate River. If this trips, your channel order is flipped.
assert class_px[0] > class_px[2] * 3, (
    "Residential <= River -- BGR/RGB channel order is probably inverted. STOP and fix before training.")
print("\nchannel-order check PASSED")

## 6. One-time repack to a 512x512 cache

Your tiles are **2345x2351 on disk (~10 MB each)**. Decoding 5.5 MP PNGs and resizing them on Kaggle's
4 vCPUs would leave the T4 idle for most of every epoch.

The repack does the resize **once**:
- **Images**: `INTER_AREA` (the correct kernel for 4.6x downsampling; bilinear aliases thin roads).
- **Masks**: colour -> class index at **full resolution first**, then `INTER_NEAREST` to 512. Resizing
  the RGB mask first would interpolate between class colours and manufacture phantom classes at every
  polygon edge.

Result: ~700 MB instead of 13 GB. Effective GSD after this step is 300 m / 512 px = **0.586 m/px**
(the figure to report); the native export is ~0.128 m/px. D4 confirmed this resize is safe for thin
structures: Road area retained 100.2%, River 100.6%, components 29->26 and 4->4, zero components under 3 px.

**Run once, then `Save Version` -> add the output as a dataset -> set `PREBUILT_CACHE` to its path** so
later sessions skip this entirely (~15-20 min saved per session).

In [ ]:
from concurrent.futures import ProcessPoolExecutor

CACHE = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)
MARKER = CACHE / "_complete.json"

def _repack_one(rec):
    _, _, iname, mname = rec
    img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)
    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)
    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    lab = LUT[pack_bgr(msk)]                                    # full-res -> class index
    lab = cv2.resize(lab, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(str(CACHE / "images" / iname), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    cv2.imwrite(str(CACHE / "labels" / mname), lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    return np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)

if MARKER.exists():
    meta = json.load(open(MARKER))
    print(f"cache present at {CACHE} ({meta['n']} pairs) -- skipping repack")
else:
    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"
    (CACHE / "images").mkdir(parents=True, exist_ok=True)
    (CACHE / "labels").mkdir(parents=True, exist_ok=True)
    t0 = time.time(); counts = {}
    with ProcessPoolExecutor(max_workers=4) as ex:
        for rec, b in zip(records, ex.map(_repack_one, records, chunksize=8)):
            counts[rec[2]] = b.tolist()
            if len(counts) % 200 == 0:
                el = time.time() - t0
                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  eta {el/len(counts)*(len(records)-len(counts)):.0f}s")
    json.dump({"n": len(counts), "counts": counts}, open(MARKER, "w"))
    print(f"repacked {len(counts)} pairs in {time.time()-t0:.0f}s")

PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in json.load(open(MARKER))["counts"].items()}
sz = sum(f.stat().st_size for f in CACHE.rglob('*.png')) / 1e9
print(f"cache size: {sz:.2f} GB")

## 7. Load the cache into RAM

1224 x (512x512x3 + 512x512) uint8 = **~1.3 GB**, comfortably inside Kaggle's ~13 GB. Holding it
resident removes disk I/O from the training loop entirely and lets `num_workers` do nothing but
augmentation. Set `PRELOAD_RAM=False` if you ever hit memory pressure.

In [ ]:
RAM = {}
if cfg.PRELOAD_RAM:
    t0 = time.time()
    for i, (_, _, iname, mname) in enumerate(records):
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        RAM[iname] = (img, lab)
        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")
    mb = sum(a.nbytes + b.nbytes for a, b in RAM.values()) / 1e6
    print(f"preloaded {len(RAM)} pairs, {mb:.0f} MB, {time.time()-t0:.0f}s")
    # sanity: labels must only ever contain 0..5 or 255
    u = np.unique(np.concatenate([b[::32, ::32].ravel() for _, b in RAM.values()]))
    assert set(u.tolist()) <= set(range(6)) | {255}, f"unexpected label values: {u}"
    print("label value check PASSED:", u.tolist())

## 8. Splitters — sequence-block proxy and random control

### The proxy, and its honest limitation
You have no surviving grid indices (the QGIS Atlas coverage layer is gone), so contiguous runs of export
order stand in for spatial blocks. QGIS Atlas iterates the coverage layer in feature order, which for a
rectangular grid is normally **row-major**. That means:

- Consecutive ids are neighbours **along the scan axis only**. Adjacency *perpendicular* to it (the tile
  directly above/below) is completely uncontrolled — those tiles can land in different folds.
- Every row wrap is a spatial discontinuity, so a "block" is really a set of strips.
- Dropped mono-class tiles break runs, so id distance overstates ground distance near gaps.

Net effect: **this proxy under-estimates true spatial leakage** relative to real geo-blocked CV. Report
it as *sequence-block CV (export-order proxy)*, not as spatial CV, and cite the random-split gap as a
lower bound on the leakage effect.

### Buffer
`BLOCK_BUFFER=2` base tiles adjacent to each held-out block are dropped from **train only** (never from
val), so the fold boundary is not two neighbouring tiles sitting on opposite sides of the split. Costs ~1%
of training data.

### Grouping
Both splitters operate on the 408 base ids; `expand()` then pulls in all three variants. A base tile and
its flips can never straddle a fold.

In [ ]:
def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

def random_split(base_ids: List[int], fold: int, n_folds: int, seed: int):
    ids = list(base_ids)
    random.Random(seed).shuffle(ids)                 # same permutation for all folds
    edges = [round(i * len(ids) / n_folds) for i in range(n_folds + 1)]
    val = ids[edges[fold]:edges[fold + 1]]
    train = ids[:edges[fold]] + ids[edges[fold + 1]:]
    return sorted(train), sorted(val)

if cfg.SPLIT_MODE == "block":
    tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)
elif cfg.SPLIT_MODE == "random":
    tr_ids, va_ids = random_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.SEED)
else:
    raise ValueError(cfg.SPLIT_MODE)

def expand(ids): return [r for i in ids for r in groups[i]]
tr_recs, va_recs = expand(tr_ids), expand(va_ids)

assert not (set(tr_ids) & set(va_ids)), "base-tile leakage between train and val"
print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}: {len(tr_ids)} train tiles / {len(va_ids)} val tiles "
      f"({len(BASE_IDS)-len(tr_ids)-len(va_ids)} buffered out)")
print(f"  -> {len(tr_recs)} train images / {len(va_recs)} val images")
print(f"  val id range: {min(va_ids)}..{max(va_ids)}")

## 9. Effective-number class weights — computed on the train split only

Weights come from the **training fold alone**. Computing them once over the full dataset would leak val
class priors into the loss; small effect, but exactly the kind of thing a Q1 reviewer flags.

### beta must scale with the counting unit
Effective number (Cui et al. 2019) is `E_c = (1 - beta^n_c) / (1 - beta)`, `w_c = 1/E_c`. The paper applies
it to **per-class image counts** (hundreds to thousands), where `beta=0.9999` is sensible because
`1/(1-beta) = 1e4` is the same order as `n_c`.

Here `n_c` is a **pixel count** (~1e6 to 1e9). At that scale `beta^n_c` underflows to 0 for *every* class,
so `E_c = 1/(1-beta)` for all c and the weights come out **exactly uniform** — plain unweighted CE, silently.
`EN_BETA_AUTO=True` selects `beta = 1 - 1/N_total` instead, putting the saturation point at the actual data
scale and recovering a meaningful spread before capping. This matches the setting resolved for all baselines.

In [ ]:
EN_BETA_AUTO = True     # protocol: True for all five baselines

def effective_number_weights(recs, cap, n_classes, beta_fixed, auto):
    n = np.zeros(n_classes, dtype=np.float64)
    for _, _, iname, _ in recs:
        n += PIX_COUNTS[iname][:n_classes]
    beta = (1.0 - 1.0 / n.sum()) if auto else beta_fixed
    eff = (1.0 - np.power(beta, n)) / (1.0 - beta)
    w = 1.0 / np.maximum(eff, 1e-12)
    w = w / w.mean()
    w = np.minimum(w, cap)
    return w / w.mean(), n, beta

CLASS_W, TRAIN_PX, BETA_USED = effective_number_weights(
    tr_recs, cfg.WEIGHT_CAP, cfg.N_CLASSES, cfg.EN_BETA, EN_BETA_AUTO)

print(f"total train pixels: {TRAIN_PX.sum():.3e} | beta used: {BETA_USED:.12f} "
      f"| saturation point 1/(1-beta) = {1/(1-BETA_USED):.3e}")
print(f"{'class':<14}{'train px %':>12}{'weight':>10}")
for i, nm in enumerate(cfg.CLASS_NAMES):
    print(f"{nm:<14}{TRAIN_PX[i]/TRAIN_PX.sum()*100:11.2f}%{CLASS_W[i]:10.3f}")
print(f"weight ratio max/min: {CLASS_W.max()/CLASS_W.min():.1f}")

if CLASS_W.max() / CLASS_W.min() < 1.05:
    print("\nWARNING: weights are effectively uniform -- beta has saturated. This is unweighted CE.")

CLASS_W_T = torch.tensor(CLASS_W, dtype=torch.float32).cuda()

## 10. Dataset and DataLoaders

**No hflip/vflip** — those are already materialised in the 1224 files. Adding them would produce duplicate
views inside a single epoch.

On-the-fly augmentation is **rot90 (k in 0..3) + mild brightness/contrast jitter**. Combined with the
baked-in flips this completes the dihedral group D4, the natural symmetry group for nadir overhead imagery,
and it introduces no new interpolation (rot90 is exact — no resampling, so 9-13 px roads and rivers stay
crisp). Scale jitter is deliberately excluded: it changes effective GSD, which would make the thin-structure
results harder to interpret.

**Do not strengthen augmentation here alone.** It is the strongest remaining anti-overfitting lever, but it
is also a protocol constant — if you add anything (cutout, colour jitter, scale), it goes into all five
notebooks or the comparison is confounded. Augmentation is seeded per (epoch, index) so the run is
reproducible and workers never collide.

In [ ]:
IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], np.float32)

class SatSeg(Dataset):
    def __init__(self, recs, train: bool):
        self.recs, self.train, self.epoch = recs, train, 0

    def __len__(self): return len(self.recs)

    def _read(self, iname, mname):
        if cfg.PRELOAD_RAM:
            img, lab = RAM[iname]
            return img.copy(), lab.copy()
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        return img, lab

    def __getitem__(self, i):
        _, _, iname, mname = self.recs[i]
        img, lab = self._read(iname, mname)

        if self.train:
            rs = np.random.RandomState((cfg.SEED * 100003 + self.epoch * 9973 + i) % (2**31))
            if cfg.AUG_ROT90:
                k = rs.randint(4)
                if k:
                    img = np.ascontiguousarray(np.rot90(img, k))
                    lab = np.ascontiguousarray(np.rot90(lab, k))
            if cfg.AUG_PHOTOMETRIC:
                a = 1.0 + rs.uniform(-cfg.AUG_CONTRAST, cfg.AUG_CONTRAST)        # contrast
                b = 255.0 * rs.uniform(-cfg.AUG_BRIGHTNESS, cfg.AUG_BRIGHTNESS)  # brightness
                img = np.clip(img.astype(np.float32) * a + b, 0, 255).astype(np.uint8)

        x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        x = (x - IMNET_MEAN) / IMNET_STD
        x = torch.from_numpy(x.transpose(2, 0, 1))
        y = torch.from_numpy(lab.astype(np.int64))
        return x, y

train_ds, val_ds = SatSeg(tr_recs, True), SatSeg(va_recs, False)
train_ld = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,
                      worker_init_fn=worker_init_fn, generator=g)
val_ld   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,
                      worker_init_fn=worker_init_fn)
xb, yb = next(iter(train_ld))
print(xb.shape, xb.dtype, yb.shape, yb.dtype, "| label values:", torch.unique(yb).tolist()[:8])

## 11. Loss — weighted CE (label-smoothed) + Lovasz-Softmax

Lovasz-Softmax is the convex surrogate of the Jaccard loss (Berman et al. 2018) and is what actually moves
mIoU on thin classes; weighted CE keeps early optimization stable. `lambda=0.75` is constant, matching
`segformer-b0-new.ipynb`.

### Label smoothing 0.05 is new, and it is the right kind of new
Your masks are CVAT polygons over 0.586 m/px imagery. A pixel one step inside a Residential polygon and a
pixel one step outside it are, in the image, nearly identical — the boundary is an annotator's line, not a
physical edge. Unsmoothed CE asks the network to drive those two pixels to opposite infinities, and the only
way to satisfy that is to memorize the individual tile. Smoothing caps the target at 0.95 and removes that
pressure. `nn.CrossEntropyLoss` applies smoothing correctly alongside `weight` and `ignore_index`.

Smoothing is applied to **CE only**, never to Lovasz — Lovasz operates on the sorted-error surrogate of a
hard set overlap, and smoothing its targets is not defined.

Two implementation details that carry over unchanged:
- The whole loss is computed in **fp32 outside autocast**. Lovasz sorts per-pixel errors and its gradient is
  a cumulative-sum difference; in fp16 that accumulates visible error and can produce NaNs.
- Ignore pixels are filtered *before* flattening, so 255 never reaches either term.

In [ ]:
def lovasz_grad(gt_sorted):
    p = len(gt_sorted); gts = gt_sorted.sum()
    inter = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jac = 1.0 - inter / union
    if p > 1: jac[1:p] = jac[1:p] - jac[0:-1]
    return jac

def lovasz_softmax_flat(probas, labels, n_classes):
    if probas.numel() == 0: return probas.sum() * 0.0
    losses = []
    for c in range(n_classes):
        fg = (labels == c).float()
        if fg.sum() == 0: continue                     # "present" classes only
        err = (fg - probas[:, c]).abs()
        err_sorted, perm = torch.sort(err, 0, descending=True)
        losses.append(torch.dot(err_sorted, lovasz_grad(fg[perm])))
    return torch.stack(losses).mean() if losses else probas.sum() * 0.0

class ComboLoss(nn.Module):
    def __init__(self, weights, lam, ignore, n_classes, label_smoothing):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weights, ignore_index=ignore,
                                      label_smoothing=label_smoothing)
        self.lam, self.ignore, self.nc = lam, ignore, n_classes

    def forward(self, logits, target):
        logits = logits.float()                        # fp32 regardless of autocast
        ce = self.ce(logits, target)
        B, C, H, W = logits.shape
        p = logits.permute(0, 2, 3, 1).reshape(-1, C).softmax(1)
        t = target.reshape(-1)
        keep = t != self.ignore
        lv = lovasz_softmax_flat(p[keep], t[keep], self.nc)
        return ce + self.lam * lv, ce.detach(), lv.detach()

criterion = ComboLoss(CLASS_W_T, cfg.LAMBDA_LOVASZ, cfg.IGNORE_INDEX,
                      cfg.N_CLASSES, cfg.LABEL_SMOOTHING).cuda()
print(f"loss ready | lambda={cfg.LAMBDA_LOVASZ} | label_smoothing={cfg.LABEL_SMOOTHING}")

## 12. Model, dropout, EMA, optimizer, scheduler

This is where almost everything changed. Four separate mechanisms, each doing a different job.

### 12.1 The network
`smp.Unet` with an ImageNet **ResNet-18** and decoder channels (128,64,32,16,8). Where the parameters went:

| block | ResNet-50 + (256,...,16) | ResNet-18 + (128,...,8) |
|---|---|---|
| encoder | 23.5 M | 11.2 M |
| decoder block 1 | 7.67 M | 1.03 M |
| decoder blocks 2-5 | 1.32 M | 0.25 M |
| **total** | **~32.5 M** | **~12.5 M** |

Decoder block 1 alone was 7.67 M — a single `Conv2d(3072, 256, 3x3)` fusing the 2048-channel bottleneck
with the 1024-channel skip. Under ResNet-18 that same conv is `Conv2d(768, 128, 3x3)`, 7.5x smaller. Most
of the decoder saving comes from one layer.

**A trap worth naming:** `encoder_depth=4` looks like the obvious way to drop ResNet's `layer4`, but smp
still *instantiates* the unused stage — it stays in `state_dict`, counts in `sum(p.numel())`, and receives
no gradient. Do not use it. The printout below is the ground truth for the params column in your table.

### 12.2 Dropout — where, and why there
`UNetDrop` wraps the smp model without renaming any parameter, so `state_dict` keys stay identical to a
plain `smp.Unet` and the inference cell in section 22 can load either.

- `Dropout2d(0.10)` on the **deepest encoder feature** (512 ch at H/32). This is the most
  over-parameterized tensor in the network and the one most able to memorize a specific tile's global
  appearance. `Dropout2d` drops whole channels rather than individual pixels — the right variant for convs,
  where neighbouring pixels are so correlated that plain elementwise dropout barely regularizes at all.
- `Dropout2d(0.10)` **before the segmentation head**, mirroring SegFormer's `classifier_dropout_prob=0.1`.

Nothing is placed in the middle of the decoder. Those layers carry the high-resolution skip information
that 9-13 px roads and rivers depend on, and dropping channels there costs thin-structure IoU directly —
which is the number you are trying to raise.

### 12.3 Weight decay that does not fight BatchNorm
`WEIGHT_DECAY=0.01` is 100x the old value, but it is applied **only to conv and linear weights**. Biases and
BatchNorm affine parameters (`ndim <= 1`) go in a `weight_decay=0` group. Decaying a BatchNorm gamma pulls
the layer's output scale toward zero, which the next layer immediately has to compensate for — it
regularizes nothing and destabilizes training. Nearly every reference implementation does this split, and
almost no student implementation does. It is a one-line thing a reviewer will notice.

### 12.4 EMA — the largest single change
`ModelEMA` keeps a shadow copy updated as `ema = d*ema + (1-d)*live` every optimizer step, with `d` ramping
in over `EMA_WARMUP_STEPS` so epoch 0 does not dominate the average forever. **Validation, early stopping,
`best.pt` and the final evaluation all use the EMA weights.** The live weights only ever take gradients.

At 51 steps/epoch and decay 0.999, the EMA has an effective window of ~1000 steps, roughly the last 20
epochs. Your old curve — val loss bottoming at epoch 8-10 then climbing to 2.8 while mIoU stayed roughly
flat — is the signature of a model oscillating around a solution rather than diverging from one. Averaging
that trajectory usually gains a point or two of mIoU for free and makes the "is it overfitting?" question
answerable instead of just visible.

### 12.5 Schedule
500 linear warmup steps, then cosine to zero over `COS_HORIZON_EPOCHS=120`, clamped so LR stays at zero if
the run somehow continues past the horizon. The old run set the cosine over `MAX_EPOCHS=200` while early
stopping fired near epoch 100 — the LR was still at ~35% of peak when training ended, so the model never got
the low-LR annealing phase where it settles into a minimum instead of bouncing around one. That, on its own,
explains a good part of the noisy late-run val loss.

In [ ]:
import segmentation_models_pytorch as smp

class UNetDrop(nn.Module):
    """smp.Unet + Dropout2d at the bottleneck and before the head.

    Submodules are re-bound (not nested), so state_dict keys are byte-identical to a plain
    smp.Unet: encoder.*, decoder.*, segmentation_head.*  Dropout carries no parameters.
    """
    def __init__(self, base: nn.Module, p_bottleneck: float, p_head: float):
        super().__init__()
        self.encoder = base.encoder
        self.decoder = base.decoder
        self.segmentation_head = base.segmentation_head
        self.drop_bottleneck = nn.Dropout2d(p_bottleneck)
        self.drop_head = nn.Dropout2d(p_head)

    def forward(self, x):
        feats = list(self.encoder(x))
        feats[-1] = self.drop_bottleneck(feats[-1])     # deepest feature, H/32
        dec = self.decoder(*feats)
        return self.segmentation_head(self.drop_head(dec))

def build_model():
    base = smp.Unet(
        encoder_name=cfg.ENCODER,
        encoder_weights=cfg.ENCODER_WEIGHTS,
        decoder_channels=list(cfg.DECODER_CHANNELS),
        decoder_attention_type=cfg.DECODER_ATTENTION,
        in_channels=3,
        classes=cfg.N_CLASSES,
    )
    return UNetDrop(base, cfg.DROPOUT_BOTTLENECK, cfg.DROPOUT_HEAD)

model = build_model().cuda()

# ---------- EMA ----------
class ModelEMA:
    def __init__(self, model, decay, warmup):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
        self.decay, self.warmup, self.updates = decay, warmup, 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = self.decay * (1.0 - math.exp(-self.updates / max(1, self.warmup)))
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(msd[k])                          # num_batches_tracked etc.

    def state_dict(self):        return self.ema.state_dict()
    def load_state_dict(self, s): self.ema.load_state_dict(s)

ema = ModelEMA(model, cfg.EMA_DECAY, cfg.EMA_WARMUP_STEPS)

# ---------- param groups: differential LR x (decay / no-decay) ----------
def split_decay(params_named, no_decay_on_norm_bias):
    decay, no_decay = [], []
    for n, p in params_named:
        if not p.requires_grad: continue
        if no_decay_on_norm_bias and (p.ndim <= 1 or n.endswith(".bias")):
            no_decay.append(p)                           # BN gamma/beta, all biases
        else:
            decay.append(p)
    return decay, no_decay

enc_named = list(model.encoder.named_parameters())
dec_named = (list(model.decoder.named_parameters())
             + list(model.segmentation_head.named_parameters()))
e_d, e_nd = split_decay(enc_named, cfg.NO_DECAY_ON_NORM_BIAS)
d_d, d_nd = split_decay(dec_named, cfg.NO_DECAY_ON_NORM_BIAS)

param_groups = [
    {"params": e_d,  "lr": cfg.LR_ENCODER, "weight_decay": cfg.WEIGHT_DECAY, "name": "encoder_decay"},
    {"params": e_nd, "lr": cfg.LR_ENCODER, "weight_decay": 0.0,              "name": "encoder_nodecay"},
    {"params": d_d,  "lr": cfg.LR_DECODER, "weight_decay": cfg.WEIGHT_DECAY, "name": "decoder_decay"},
    {"params": d_nd, "lr": cfg.LR_DECODER, "weight_decay": 0.0,              "name": "decoder_nodecay"},
]
optimizer = torch.optim.AdamW(param_groups)
DEC_GROUP_IDX = [i for i, g in enumerate(optimizer.param_groups) if g["name"] == "decoder_decay"][0]

# ---------- schedule: warmup then cosine over COS_HORIZON_EPOCHS ----------
steps_per_epoch = len(train_ld) // cfg.GRAD_ACCUM
cosine_steps    = steps_per_epoch * cfg.COS_HORIZON_EPOCHS

def lr_lambda(step):
    if step < cfg.WARMUP_STEPS:
        return (step + 1) / cfg.WARMUP_STEPS
    prog = (step - cfg.WARMUP_STEPS) / max(1, cosine_steps - cfg.WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))   # clamped: stays 0 past the horizon

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = make_scaler(enabled=cfg.AMP)

# ---------- parameter accounting (this is the params column in the paper) ----------
n_enc  = sum(p.numel() for p in model.encoder.parameters())
n_dec  = sum(p.numel() for p in model.decoder.parameters())
n_head = sum(p.numel() for p in model.segmentation_head.parameters())
n_tot  = sum(p.numel() for p in model.parameters())
print(f"PARAMS  total {n_tot/1e6:.2f} M  =  encoder {n_enc/1e6:.2f} M "
      f"+ decoder {n_dec/1e6:.2f} M + head {n_head/1e3:.1f} K")
print(f"decay groups: {sum(p.numel() for p in e_d+d_d)/1e6:.2f} M decayed, "
      f"{sum(p.numel() for p in e_nd+d_nd)/1e3:.1f} K exempt (norm/bias)")
print(f"steps/epoch {steps_per_epoch} | warmup {cfg.WARMUP_STEPS} "
      f"| cosine over {cosine_steps} steps ({cfg.COS_HORIZON_EPOCHS} epochs) | max {cfg.MAX_EPOCHS} epochs")

def arch_signature(m):
    """Stamped into every checkpoint; section 15 refuses to load a mismatch."""
    sd = m.state_dict()
    return {"arch": cfg.ARCH_VERSION, "n_keys": len(sd),
            "n_params": int(sum(v.numel() for v in sd.values())),
            "encoder": cfg.ENCODER, "decoder_channels": list(cfg.DECODER_CHANNELS)}

ARCH_SIG = arch_signature(model)
print("arch signature:", ARCH_SIG)

## 13. Metrics module

Everything accumulates into a single 6x6 confusion matrix built on-GPU with `bincount` (fast enough to run
every epoch). Ignore pixels are dropped before accumulation, so 255 never enters any metric.

- **IoU / F1 / mIoU / mF1** — macro over the 6 classes
- **OA**, **frequency-weighted IoU**
- **Cohen's kappa** — from the same matrix
- **Producer's accuracy** = recall (row-normalized), **User's accuracy** = precision (column-normalized).
  These are the remote-sensing names for the same quantities; reporting both plus the normalized matrix is
  what reviewers in this field expect.
- **clDice** (Shit et al. 2021) for **Road** and **River** only — a topology-aware score that rewards
  connectivity rather than area overlap, the right lens for linear features at 9-13 px. Skeletonization is
  CPU-bound, so it runs at final evaluation only, not every epoch.

In [ ]:
from skimage.morphology import skeletonize

class ConfMat:
    def __init__(self, n): self.n = n; self.m = torch.zeros(n, n, dtype=torch.int64, device="cuda")
    def update(self, pred, tgt):
        k = tgt != cfg.IGNORE_INDEX
        if k.sum() == 0: return
        self.m += torch.bincount(self.n * tgt[k].long() + pred[k].long(),
                                 minlength=self.n ** 2).reshape(self.n, self.n)
    def numpy(self): return self.m.cpu().numpy().astype(np.float64)

def metrics_from_cm(cm, names):
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tot = cm.sum()
    iou = tp / np.maximum(tp + fp + fn, 1e-12)
    f1  = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)
    prod = tp / np.maximum(cm.sum(1), 1e-12)      # recall
    user = tp / np.maximum(cm.sum(0), 1e-12)      # precision
    freq = cm.sum(1) / max(tot, 1e-12)
    po = tp.sum() / max(tot, 1e-12)
    pe = (cm.sum(0) * cm.sum(1)).sum() / max(tot ** 2, 1e-12)
    return {
        "mIoU": float(np.nanmean(iou)), "mF1": float(np.nanmean(f1)),
        "OA": float(po), "FWIoU": float((freq * iou).sum()),
        "kappa": float((po - pe) / max(1 - pe, 1e-12)),
        "per_class": {n: {"IoU": float(iou[i]), "F1": float(f1[i]),
                          "producer_acc": float(prod[i]), "user_acc": float(user[i]),
                          "support_px": int(cm.sum(1)[i])} for i, n in enumerate(names)},
        "confusion_matrix_raw": cm.astype(np.int64).tolist(),
        "confusion_matrix_norm": (cm / np.maximum(cm.sum(1, keepdims=True), 1e-12)).tolist(),
    }

def cldice_pair(pred_bin, gt_bin):
    if gt_bin.sum() == 0 and pred_bin.sum() == 0: return None
    sp = skeletonize(pred_bin.astype(bool)); sg = skeletonize(gt_bin.astype(bool))
    tprec = (sp & gt_bin.astype(bool)).sum() / max(sp.sum(), 1)     # topology precision
    tsens = (sg & pred_bin.astype(bool)).sum() / max(sg.sum(), 1)   # topology sensitivity
    if tprec + tsens == 0: return 0.0
    return float(2 * tprec * tsens / (tprec + tsens))

## 14. Efficiency metrics

Params, GFLOPs at 512x512, peak training VRAM, wall-clock, and inference FPS. FPS is measured at batch 1
with 20 warmup iterations and CUDA synchronization — without the warmup you are timing cuDNN autotuning,
and on a shared Kaggle T4 you should treat FPS as indicative rather than exact.

Measured on the **EMA model in eval mode**, so dropout is inactive and the numbers describe the network you
would actually deploy. Use this same code for all five baselines so the column is internally consistent.

In [ ]:
def efficiency_report(m):
    from ptflops import get_model_complexity_info
    m.eval()
    with amp_autocast(enabled=False):
        macs, params = get_model_complexity_info(
            m, (3, cfg.IMG_SIZE, cfg.IMG_SIZE),
            as_strings=False, print_per_layer_stat=False, verbose=False)
    x = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda")
    with torch.no_grad(), amp_autocast(enabled=cfg.AMP):
        for _ in range(20): m(x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(50): m(x)
        torch.cuda.synchronize()
    return {"params_M": params / 1e6,
            "GFLOPs_512": macs * 2 / 1e9,           # ptflops returns MACs; FLOPs = 2 x MACs
            "inference_fps_bs1": 50 / (time.time() - t0)}

## 15. Hugging Face persistence, resume, and the architecture guard

Each fold gets its **own repo** (`geonusaf-unet-r18-{mode}-fold{k}`) so any single fold can be pulled
independently for inference later — nothing is ever overwritten across folds.

Two artifacts per repo:
- `best.pt` — **EMA weights** + config + metrics + arch signature, pushed whenever val mIoU improves. This
  is what the inference script downloads.
- `last.pt` — full state (live weights, EMA, optimizer, scaler, scheduler, history), pushed every
  `PUSH_LAST_EVERY` epochs. This is what a new session resumes from.

### The architecture guard
You lost a TC-SegFormer session when a repo still held a checkpoint from an older architecture and
`load_state_dict` blew up mid-resume. `check_arch` compares the stored signature against the live model
before any load: on mismatch it deletes the stale `best.pt`/`last.pt` from the Hub and starts fresh instead
of crashing. Since this notebook replaces the ResNet-50 run, **use a new `HF_PREFIX`** (already set to
`geonusaf-unet-r18`) so the old ResNet-50 repos stay intact as the second row of your capacity ablation.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)

def hf_push(local_path, name, msg):
    api.upload_file(path_or_fileobj=local_path, path_in_repo=name,
                    repo_id=HF_REPO, token=HF_TOKEN, commit_message=msg)

def hf_try_download(name):
    try: return hf_hub_download(HF_REPO, name, token=HF_TOKEN)
    except Exception: return None

def hf_delete(name):
    try:
        api.delete_file(path_in_repo=name, repo_id=HF_REPO, token=HF_TOKEN,
                        commit_message=f"drop stale {name} (arch mismatch)")
        print(f"  deleted stale {name} from {HF_REPO}")
    except Exception as e:
        print(f"  could not delete {name}: {type(e).__name__}")

def check_arch(ck_sig) -> bool:
    """True if the checkpoint was written by this exact architecture."""
    if not isinstance(ck_sig, dict): return False
    for k in ("arch", "n_keys", "n_params", "encoder", "decoder_channels"):
        if ck_sig.get(k) != ARCH_SIG[k]:
            print(f"  ARCH MISMATCH on '{k}': checkpoint={ck_sig.get(k)} live={ARCH_SIG[k]}")
            return False
    return True

def push_model_card(best_metrics, epoch):
    pc = best_metrics.get("per_class", {})
    L = []
    L.append("---")
    L.append("tags: [semantic-segmentation, remote-sensing, unet, resnet18, geonusaf]")
    L.append("library_name: segmentation-models-pytorch")
    L.append("---")
    L.append("")
    L.append(f"# GeoNUSAF - {cfg.MODEL_TAG} - {cfg.SPLIT_MODE} split, fold {cfg.FOLD}")
    L.append("")
    L.append("Kathmandu Valley land-use segmentation, 6 classes, ignore_index=255.")
    L.append("Weights are the **EMA** weights (decay 0.999), not the raw final weights.")
    L.append("")
    L.append("| field | value |")
    L.append("|---|---|")
    L.append(f"| architecture | smp.Unet, encoder {cfg.ENCODER} (ImageNet), decoder {list(cfg.DECODER_CHANNELS)} |")
    L.append(f"| params | {n_tot/1e6:.2f} M |")
    L.append(f"| arch version | {cfg.ARCH_VERSION} |")
    L.append(f"| split mode | {cfg.SPLIT_MODE} |")
    L.append(f"| fold | {cfg.FOLD} of {cfg.N_FOLDS} |")
    L.append(f"| seed | {cfg.SEED} |")
    L.append("| input | 512x512, ImageNet norm, effective GSD 0.586 m/px |")
    L.append(f"| regularization | wd {cfg.WEIGHT_DECAY} (norm/bias exempt), ls {cfg.LABEL_SMOOTHING}, "
             f"drop {cfg.DROPOUT_HEAD}, EMA {cfg.EMA_DECAY} |")
    L.append(f"| classes | {', '.join(cfg.CLASS_NAMES)} |")
    L.append(f"| best epoch | {epoch} |")
    L.append(f"| val mIoU | {best_metrics.get('mIoU', float('nan')):.4f} |")
    L.append(f"| val mF1 | {best_metrics.get('mF1', float('nan')):.4f} |")
    L.append(f"| val OA | {best_metrics.get('OA', float('nan')):.4f} |")
    L.append(f"| val kappa | {best_metrics.get('kappa', float('nan')):.4f} |")
    L.append("")
    L.append("## Per-class (validation)")
    L.append("| class | IoU | F1 |")
    L.append("|---|---|---|")
    for k, v in pc.items():
        L.append(f"| {k} | {v['IoU']:.4f} | {v['F1']:.4f} |")
    L.append("")
    L.append("Checkpoint `best.pt` holds `model_state` (EMA) plus `cfg`, `metrics` and `arch_sig`.")
    card = "\n".join(L)
    p = f"{cfg.OUT_DIR}/README.md"; open(p, "w").write(card)
    hf_push(p, "README.md", f"card epoch {epoch}")

print("HF ready:", HF_REPO)

## 16. Weights & Biases

Matches the logging on the other baselines. `resume="allow"` with a deterministic run id means a resumed
session continues the same W&B run rather than starting a second one. The config it uploads includes the
whole regularization block, so you can diff two runs in the W&B UI and see exactly which knob differed.

In [ ]:
import wandb
try:
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB = True
except Exception as e:
    print("wandb disabled:", e); WANDB = False

if WANDB:
    wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME,
               resume="allow", config={**asdict(cfg), "params_M": n_tot / 1e6})

## 17. Train / validate loops

- `scaler.unscale_(optimizer)` runs **before** `clip_grad_norm_`, otherwise you would be clipping
  loss-scaled gradients and the 1.0 threshold would mean nothing. `clip_grad_norm_` returns the pre-clip
  norm, which is what gets logged.
- `ema.update(model)` fires **after every optimizer step**, not every batch — under gradient accumulation
  those differ, and updating per batch would silently change the effective EMA window.
- **`validate()` takes the module to evaluate.** The main loop calls it twice: once on the EMA model (the
  reported numbers, used for early stopping and checkpointing) and once on the live model. Logging both is
  cheap and it is what lets you *demonstrate* in the paper that EMA helped, rather than asserting it.
- Validation runs at full 512x512 with no TTA. Keep it that way across all baselines; TTA belongs in a
  separate row of the table, not in the protocol.
- Early stopping tracks **val mIoU**, not val loss — the loss is class-weighted and label-smoothed, so it
  moves for reasons mIoU does not care about. With smoothing, val loss has a nonzero floor and will look
  worse than your old run's at the same quality; do not read anything into its absolute value.

In [ ]:
def train_one_epoch(epoch):
    model.train(); train_ds.epoch = epoch
    tot, n, gn_acc = 0.0, 0, []
    optimizer.zero_grad(set_to_none=True)
    for it, (x, y) in enumerate(train_ld):
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with amp_autocast(enabled=cfg.AMP):
            logits = model(x)
        loss, ce, lv = criterion(logits, y)
        scaler.scale(loss / cfg.GRAD_ACCUM).backward()

        if (it + 1) % cfg.GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)                                  # unscale BEFORE clipping
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)
            gn_acc.append(float(gn))
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            ema.update(model)                                           # one EMA step per optimizer step

        tot += float(loss) * x.size(0); n += x.size(0)
        if it % 20 == 0:
            print(f"  e{epoch} it{it}/{len(train_ld)} loss {float(loss):.4f} "
                  f"(ce {float(ce):.4f} lov {float(lv):.4f}) gnorm {gn_acc[-1] if gn_acc else 0:.3f}",
                  flush=True)
    return tot / max(n, 1), float(np.mean(gn_acc)), float(np.max(gn_acc))

@torch.no_grad()
def validate(net):
    net.eval(); cm = ConfMat(cfg.N_CLASSES); tot, n = 0.0, 0
    for x, y in val_ld:
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with amp_autocast(enabled=cfg.AMP):
            logits = net(x)
        loss, _, _ = criterion(logits, y)
        cm.update(logits.float().argmax(1), y)
        tot += float(loss) * x.size(0); n += x.size(0)
    return tot / max(n, 1), metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)

## 18. Main loop with automatic resume

On start this pulls `last.pt` from the fold's HF repo, verifies the architecture signature, and continues
from that epoch. So "what if it exceeds 12 hours" is answered by: it stops, you start a new session, run the
notebook top to bottom unchanged, and it picks up where it left off.

### Reading the log
Three columns tell you whether the overfitting is fixed:
- **`mIoU_ema` vs `mIoU_raw`** — if EMA is above raw, the model was oscillating and averaging helped. Expect
  a gap of roughly +0.01 to +0.03 in the middle of training that narrows as the cosine anneals.
- **`gen_gap`** = val loss minus train loss. This is the overfitting number to *report*, not to panic about.
  It will grow; what matters is that it grows slowly and that `mIoU_ema` is still rising while it does.
- **`bad`** — epochs since the last improvement that cleared `MIN_DELTA`. Under the old any-improvement rule
  this counter kept getting reset by +0.0001 blips and the run drifted long past its real peak.

**Expected cost.** ~816 train images, batch 16, 51 steps/epoch. ResNet-18 at 512 with the cache in RAM and
`cudnn.benchmark` on should land around 20-35 s/epoch including both validation passes — call it 1-2 h for
a full 200 epochs, well inside one session. That is roughly a third of what the ResNet-50 run cost, and most
of the saving is the determinism fix from section 3 rather than the smaller encoder.

In [ ]:
start_epoch, best_miou, best_epoch, bad, history = 0, -1.0, -1, 0, []
ref_miou, best_metrics = -1.0, {}

ck = hf_try_download("last.pt") if cfg.RESUME else None
if ck:
    st = torch.load(ck, map_location="cuda", weights_only=False)
    if check_arch(st.get("arch_sig")):
        model.load_state_dict(st["model_state"])
        ema.load_state_dict(st["ema_state"]); ema.updates = st.get("ema_updates", 0)
        optimizer.load_state_dict(st["optim_state"])
        scheduler.load_state_dict(st["sched_state"]); scaler.load_state_dict(st["scaler_state"])
        start_epoch = st["epoch"] + 1
        best_miou, best_epoch = st["best_miou"], st["best_epoch"]
        bad, history = st["bad"], st["history"]
        ref_miou = st.get("ref_miou", best_miou)
        print(f"RESUMED from epoch {st['epoch']} (best mIoU {best_miou:.4f} @ e{best_epoch}, "
              f"ema updates {ema.updates})")
    else:
        print("stale checkpoint from a different architecture -- discarding and starting fresh")
        hf_delete("last.pt"); hf_delete("best.pt")
else:
    print("fresh start")

torch.cuda.reset_peak_memory_stats()
t_start = time.time()

for epoch in range(start_epoch, cfg.MAX_EPOCHS):
    te = time.time()
    tr_loss, gn_mean, gn_max = train_one_epoch(epoch)
    va_loss, m      = validate(ema.ema)        # EMA = the reported model
    va_loss_r, m_r  = validate(model)          # raw weights, logged for the EMA ablation
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    row = {"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
           "val_loss_raw": va_loss_r, "gen_gap": va_loss - tr_loss,
           "val_mIoU": m["mIoU"], "val_mIoU_raw": m_r["mIoU"],
           "ema_gain": m["mIoU"] - m_r["mIoU"],
           "val_mF1": m["mF1"], "val_OA": m["OA"], "val_kappa": m["kappa"],
           "grad_norm_mean": gn_mean, "grad_norm_max": gn_max,
           "lr_dec": scheduler.get_last_lr()[DEC_GROUP_IDX], "peak_vram_gb": peak_gb,
           "bad": bad, "epoch_sec": time.time() - te,
           **{f"IoU/{k}": v["IoU"] for k, v in m["per_class"].items()}}
    history.append(row)
    if WANDB: wandb.log(row, step=epoch)
    print(f"[e{epoch}] train {tr_loss:.4f} | val {va_loss:.4f} | gap {row['gen_gap']:+.3f} | "
          f"mIoU ema {m['mIoU']:.4f} raw {m_r['mIoU']:.4f} ({row['ema_gain']:+.4f}) | "
          f"gnorm mu{gn_mean:.2f}/max{gn_max:.2f} | bad {bad} | {row['epoch_sec']:.0f}s | {peak_gb:.1f}GB")

    # best.pt saves on ANY improvement; the patience counter uses MIN_DELTA
    if m["mIoU"] > best_miou:
        best_miou, best_epoch, best_metrics = m["mIoU"], epoch, m
        p = f"{cfg.OUT_DIR}/best.pt"
        torch.save({"model_state": ema.state_dict(), "arch_sig": ARCH_SIG,
                    "cfg": asdict(cfg), "metrics": m, "epoch": epoch,
                    "metrics_raw": m_r}, p)
        hf_push(p, "best.pt", f"best mIoU {best_miou:.4f} @ e{epoch}")
        push_model_card(best_metrics, epoch)
        print(f"  -> new best {best_miou:.4f} (EMA), pushed")

    if m["mIoU"] > ref_miou + cfg.MIN_DELTA:
        ref_miou, bad = m["mIoU"], 0
    else:
        bad += 1

    if (epoch % cfg.PUSH_LAST_EVERY == 0) or (bad >= cfg.PATIENCE) or (epoch == cfg.MAX_EPOCHS - 1):
        p = f"{cfg.OUT_DIR}/last.pt"
        torch.save({"model_state": model.state_dict(), "ema_state": ema.state_dict(),
                    "ema_updates": ema.updates, "optim_state": optimizer.state_dict(),
                    "sched_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
                    "arch_sig": ARCH_SIG, "epoch": epoch, "best_miou": best_miou,
                    "best_epoch": best_epoch, "ref_miou": ref_miou, "bad": bad,
                    "history": history}, p)
        hf_push(p, "last.pt", f"state e{epoch}")

    if bad >= cfg.PATIENCE:
        print(f"EARLY STOP at epoch {epoch} (best {best_miou:.4f} @ e{best_epoch}, "
              f"no +{cfg.MIN_DELTA} gain in {bad} epochs)"); break

wall_h = (time.time() - t_start) / 3600
print(f"training done | best mIoU {best_miou:.4f} @ epoch {best_epoch} | this session {wall_h:.2f} h")

## 19. Final evaluation of the best checkpoint

Reloads `best.pt` (the **EMA** weights) into a fresh model and computes the full metric suite on the
held-out fold, including clDice for Road and River and the efficiency block. clDice is averaged over images
in which the class is present in GT or prediction; images where the class is absent from both are skipped
rather than scored as 1.0, which would badly inflate the number given how sparse River is (0.57% of pixels).

`eval_model` is a separate object from `model`, so nothing downstream accidentally evaluates the live
weights. It is in `eval()` mode, so both dropout layers are inactive.

In [ ]:
eval_model = build_model().cuda()
_ck = torch.load(f"{cfg.OUT_DIR}/best.pt", map_location="cuda", weights_only=False)
assert check_arch(_ck["arch_sig"]), "best.pt architecture does not match the live model"
eval_model.load_state_dict(_ck["model_state"])
eval_model.eval()
print(f"loaded best.pt from epoch {_ck['epoch']} (EMA weights)")

cm = ConfMat(cfg.N_CLASSES)
cld = {1: [], 2: []}                                   # Road, River
with torch.no_grad():
    for x, y in val_ld:
        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)
        with amp_autocast(enabled=cfg.AMP):
            logits = eval_model(x)
        pred = logits.float().argmax(1)
        cm.update(pred, y)
        pn, yn = pred.cpu().numpy(), y.cpu().numpy()
        valid = yn != cfg.IGNORE_INDEX
        for b in range(pn.shape[0]):
            for c in cld:
                pb = (pn[b] == c) & valid[b]
                gb = (yn[b] == c) & valid[b]
                v = cldice_pair(pb, gb)
                if v is not None: cld[c].append(v)

final = metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)
final["clDice"] = {cfg.CLASS_NAMES[c]: (float(np.mean(v)) if v else None) for c, v in cld.items()}
final["clDice_n_images"] = {cfg.CLASS_NAMES[c]: len(v) for c, v in cld.items()}
final.update(efficiency_report(eval_model))
final["params_M_counted"] = n_tot / 1e6
final["peak_train_vram_gb"] = float(torch.cuda.max_memory_allocated() / 1e9)
final["train_wall_clock_h_this_session"] = wall_h
final["best_epoch"] = best_epoch
final["best_mIoU_raw_at_best_epoch"] = _ck.get("metrics_raw", {}).get("mIoU")
final["run"] = {"model": cfg.MODEL_TAG, "arch_version": cfg.ARCH_VERSION,
                "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "seed": cfg.SEED,
                "weights": "EMA"}

print(json.dumps({k: v for k, v in final.items()
                  if k not in ("confusion_matrix_raw", "confusion_matrix_norm", "per_class")}, indent=2))
print("\nper class:")
for k, v in final["per_class"].items():
    print(f"  {k:<14} IoU {v['IoU']:.4f}  F1 {v['F1']:.4f}  PA {v['producer_acc']:.4f}  UA {v['user_acc']:.4f}")

## 20. Persist results and plot the overfitting curves

Writes `results_{mode}_fold{k}.json` (full, incl. confusion matrices), a flat `.csv` row for the comparison
table, and `history.csv` for the curves, then pushes all three plus the figure to the fold's HF repo.

The figure is the direct answer to "did the regularization work". Panel 1 is train vs val loss — compare the
shape to your ResNet-50 run, where val bottomed at epoch 8-10 and climbed to 2.8. Panel 2 is the EMA vs raw
mIoU gap, which is the evidence for keeping EMA in the protocol. Panel 3 is `gen_gap` over time; a straight
rising line means capacity, a line that flattens means the regularization caught it.

In [ ]:
import csv
import matplotlib.pyplot as plt

tag = f"{cfg.SPLIT_MODE}_fold{cfg.FOLD}"
jp = f"{cfg.OUT_DIR}/results_{tag}.json"
json.dump(final, open(jp, "w"), indent=2)

flat = {"model": cfg.MODEL_TAG, "arch_version": cfg.ARCH_VERSION, "split_mode": cfg.SPLIT_MODE,
        "fold": cfg.FOLD, "seed": cfg.SEED,
        "mIoU": final["mIoU"], "mF1": final["mF1"], "OA": final["OA"],
        "FWIoU": final["FWIoU"], "kappa": final["kappa"],
        "params_M": final["params_M"], "GFLOPs_512": final["GFLOPs_512"],
        "fps_bs1": final["inference_fps_bs1"], "peak_vram_gb": final["peak_train_vram_gb"],
        "best_epoch": final["best_epoch"],
        "mIoU_raw_at_best": final["best_mIoU_raw_at_best_epoch"]}
for k, v in final["per_class"].items():
    flat[f"IoU_{k}"] = v["IoU"]; flat[f"F1_{k}"] = v["F1"]
for k, v in final["clDice"].items():
    flat[f"clDice_{k}"] = v

cp = f"{cfg.OUT_DIR}/results_{tag}.csv"
with open(cp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(flat)); w.writeheader(); w.writerow(flat)

hp = f"{cfg.OUT_DIR}/history_{tag}.csv"
with open(hp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0])); w.writeheader(); w.writerows(history)

# ---- overfitting diagnostic figure ----
ep = [r["epoch"] for r in history]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(ep, [r["train_loss"] for r in history], label="train")
ax[0].plot(ep, [r["val_loss"] for r in history], label="val (EMA)")
ax[0].plot(ep, [r["val_loss_raw"] for r in history], label="val (raw)", alpha=0.5)
ax[0].axvline(best_epoch, ls="--", c="k", lw=0.8)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title("loss"); ax[0].legend(fontsize=8)

ax[1].plot(ep, [r["val_mIoU"] for r in history], label="val mIoU (EMA)")
ax[1].plot(ep, [r["val_mIoU_raw"] for r in history], label="val mIoU (raw)", alpha=0.6)
ax[1].axvline(best_epoch, ls="--", c="k", lw=0.8)
ax[1].set_xlabel("epoch"); ax[1].set_title(f"mIoU (best {best_miou:.4f} @ e{best_epoch})")
ax[1].legend(fontsize=8)

ax[2].plot(ep, [r["gen_gap"] for r in history], c="crimson")
ax[2].axhline(0, c="k", lw=0.6); ax[2].axvline(best_epoch, ls="--", c="k", lw=0.8)
ax[2].set_xlabel("epoch"); ax[2].set_title("generalization gap (val - train loss)")

fig.suptitle(f"{cfg.MODEL_TAG} ({n_tot/1e6:.1f} M) - {cfg.SPLIT_MODE} fold {cfg.FOLD}")
plt.tight_layout()
fp = f"{cfg.OUT_DIR}/curves_{tag}.png"
plt.savefig(fp, dpi=140, bbox_inches="tight"); plt.show()

for p, n in [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"),
             (hp, f"history_{tag}.csv"), (fp, f"curves_{tag}.png")]:
    hf_push(p, n, "results")
if WANDB: wandb.finish()
print("results pushed to", HF_REPO)

## 21. Aggregation across folds — run after all folds finish

Pulls each fold's CSV from its own HF repo and reports mean +/- std per split mode. The **block-mode mIoU
minus random-mode mIoU is the leakage gap** — the headline number justifying why the block protocol is the
one you report. Expect random to score higher; if the gap is near zero, that is itself a finding worth
stating (it would suggest the export-order proxy is not capturing much spatial structure, consistent with
the row-major limitation noted in section 8).

Set `COMPARE_PREFIX` to the ResNet-50 run's HF prefix to print the capacity ablation side by side.

In [ ]:
import pandas as pd

COMPARE_PREFIX = None    # e.g. "geonusaf-unet-r50" to print the capacity ablation alongside

def pull(prefix):
    rows = []
    for mode in ["block", "random"]:
        for k in range(cfg.N_FOLDS):
            repo = f"{cfg.HF_USER}/{prefix}-{mode}-fold{k}"
            try:
                rows.append(pd.read_csv(hf_hub_download(repo, f"results_{mode}_fold{k}.csv", token=HF_TOKEN)))
            except Exception as e:
                print(f"  missing {prefix} {mode} fold {k}: {type(e).__name__}")
    return pd.concat(rows, ignore_index=True) if rows else None

df = pull(cfg.HF_PREFIX)
if df is not None:
    df.to_csv(f"{cfg.OUT_DIR}/aggregate_{cfg.HF_PREFIX}.csv", index=False)
    num = df.select_dtypes("number").columns
    agg = df.groupby("split_mode")[num].agg(["mean", "std"])
    print(agg[[c for c in [("mIoU","mean"),("mIoU","std"),("mF1","mean"),("kappa","mean")] if c in agg]])
    if {"block", "random"} <= set(df.split_mode):
        b = df[df.split_mode == "block"].mIoU.mean(); r = df[df.split_mode == "random"].mIoU.mean()
        print(f"\nLEAKAGE GAP (random - block) mIoU: {r-b:+.4f}  [block {b:.4f} | random {r:.4f}]")

if COMPARE_PREFIX:
    old = pull(COMPARE_PREFIX)
    if old is not None and df is not None:
        both = pd.concat([old.assign(run=COMPARE_PREFIX), df.assign(run=cfg.HF_PREFIX)], ignore_index=True)
        print("\nCAPACITY ABLATION (block folds only)")
        blk = both[both.split_mode == "block"]
        print(blk.groupby("run")[["params_M", "mIoU", "mF1", "kappa", "best_epoch"]].mean().round(4))

## 22. Standalone inference from Hugging Face

Self-contained: point `INFER_REPO` at any fold's repo, give it images, get predictions. Nothing above this
cell is required — it rebuilds the model from `best.pt`'s embedded config, so you can paste it into a fresh
notebook. Dropout modules carry no parameters, so the checkpoint also loads into a plain `smp.Unet` if you
ever want to strip the wrapper.

Inputs are raw full-resolution tiles; the same `INTER_AREA` downsample as training is applied so inference
matches training preprocessing exactly.

In [ ]:
def load_from_hf(repo_id, token=None, device="cuda"):
    import segmentation_models_pytorch as smp
    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token),
                    map_location=device, weights_only=False)
    c = ck["cfg"]
    base = smp.Unet(encoder_name=c["ENCODER"], encoder_weights=None,
                    decoder_channels=list(c["DECODER_CHANNELS"]),
                    decoder_attention_type=c["DECODER_ATTENTION"],
                    in_channels=3, classes=c["N_CLASSES"])
    m = UNetDrop(base, c["DROPOUT_BOTTLENECK"], c["DROPOUT_HEAD"]).to(device).eval()
    m.load_state_dict(ck["model_state"])          # EMA weights
    return m, c, ck["metrics"]

@torch.no_grad()
def predict_paths(model_, c, paths, device="cuda", batch=8):
    mean = np.array([0.485, 0.456, 0.406], np.float32); std = np.array([0.229, 0.224, 0.225], np.float32)
    out = []
    for i in range(0, len(paths), batch):
        xs = []
        for p in paths[i:i + batch]:
            im = cv2.imread(str(p), cv2.IMREAD_COLOR)
            im = cv2.resize(im, (c["IMG_SIZE"], c["IMG_SIZE"]), interpolation=cv2.INTER_AREA)
            a = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            xs.append(torch.from_numpy(((a - mean) / std).transpose(2, 0, 1)))
        x = torch.stack(xs).to(device)
        with amp_autocast(enabled=True):
            out.append(model_(x).float().argmax(1).cpu().numpy())
    return np.concatenate(out)

# --- example ---
INFER_REPO = HF_REPO
inf_model, inf_cfg, inf_metrics = load_from_hf(INFER_REPO, token=HF_TOKEN)
print(f"loaded {INFER_REPO} | val mIoU {inf_metrics['mIoU']:.4f}")
demo = [IMG_DIR / r[2] for r in va_recs[:4]]
preds = predict_paths(inf_model, inf_cfg, demo)
print("pred shape:", preds.shape, "| classes present:", np.unique(preds).tolist())

## 23. Qualitative triptychs — image / GT / prediction

The palette is your BGR table reversed to RGB for matplotlib; ignore renders black. The selection is
deliberately not random: it shows the highest- and lowest-mIoU validation tiles, because the failure cases
are what a reviewer wants to see. Watch specifically for the confusions D1 quantified — Residential ->
Agricultural (0.247), Road -> Agricultural (0.227), River -> UnusedLand (0.280), UnusedLand -> Forest
(0.441) — plus Road fragmentation, which is where clDice and IoU will disagree.

Predictions come from `eval_model` (EMA, eval mode), so these figures show the same network the metrics
describe.

In [ ]:
from matplotlib.patches import Patch

PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]                       # BGR table -> RGB for matplotlib
PALETTE_RGB[255] = (0, 0, 0)

def colorize(lab): return PALETTE_RGB[lab]

@torch.no_grad()
def per_image_miou(recs, k=3):
    scores = []
    idx_of = {r: i for i, r in enumerate(val_ds.recs)}
    for j in range(0, len(recs), 16):
        chunk = recs[j:j + 16]
        xs, ys = zip(*[val_ds[idx_of[r]] for r in chunk])
        x = torch.stack(xs).cuda(); y = torch.stack(ys).cuda()
        with amp_autocast(enabled=cfg.AMP):
            p = eval_model(x).float().argmax(1)
        for b, r in enumerate(chunk):
            c = ConfMat(cfg.N_CLASSES); c.update(p[b], y[b])
            scores.append((metrics_from_cm(c.numpy(), cfg.CLASS_NAMES)["mIoU"], r))
    scores.sort(key=lambda t: t[0])
    return scores[-k:][::-1], scores[:k]

best_imgs, worst_imgs = per_image_miou(va_recs, k=3)

def triptych(items, title):
    idx_of = {r: i for i, r in enumerate(val_ds.recs)}
    fig, axes = plt.subplots(len(items), 3, figsize=(13, 4.3 * len(items)))
    axes = np.atleast_2d(axes)
    for r, (score, rec) in enumerate(items):
        _, _, iname, mname = rec
        img = cv2.cvtColor(cv2.imread(str(CACHE / "images" / iname)), cv2.COLOR_BGR2RGB)
        gt = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        x, _ = val_ds[idx_of[rec]]
        with torch.no_grad(), amp_autocast(enabled=cfg.AMP):
            pr = eval_model(x[None].cuda()).float().argmax(1)[0].cpu().numpy().astype(np.uint8)
        pr_vis = np.where(gt == 255, 255, pr).astype(np.uint8)   # blank ignore for fair viewing
        for c, (im, t) in enumerate([(img, iname), (colorize(gt), "ground truth"),
                                     (colorize(pr_vis), f"prediction (mIoU {score:.3f})")]):
            axes[r, c].imshow(im); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis("off")
    handles = [Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)]
    handles.append(Patch(color="black", label="ignore/unannotated"))
    fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=8, frameon=False)
    fig.suptitle(f"{title} - {cfg.MODEL_TAG}, {cfg.SPLIT_MODE} fold {cfg.FOLD}", fontsize=12)
    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    p = f"{cfg.OUT_DIR}/qual_{title.replace(' ','_')}_{tag}.png"
    plt.savefig(p, dpi=130, bbox_inches="tight"); plt.show()
    hf_push(p, os.path.basename(p), "qualitative")

triptych(best_imgs, "best cases")
triptych(worst_imgs, "worst cases")

## 24. Confusion matrix figure

In [ ]:
cmn = np.array(final["confusion_matrix_norm"])
fig, ax = plt.subplots(figsize=(7.2, 6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(cfg.CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Row-normalized confusion - {cfg.MODEL_TAG}, {cfg.SPLIT_MODE} fold {cfg.FOLD}")
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cmn[i, j] > 0.5 else "black")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
p = f"{cfg.OUT_DIR}/confusion_{tag}.png"
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
hf_push(p, os.path.basename(p), "confusion matrix")

## 25. Run checklist and cross-baseline sync

### This notebook
1. Internet **ON**; Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`; set `cfg.HF_USER`.
2. Section 5 must print **channel-order check PASSED** before you go further.
3. Section 12 must print **~12.5 M** total params. If it prints ~14 M you left `DECODER_CHANNELS` at the
   old value; if ~32 M, `ENCODER` is still `resnet50`.
4. First session: let section 6 repack (~15-20 min), then `Save Version`, attach the output as a dataset,
   and set `PREBUILT_CACHE` for every later session.
5. Run the six fold-runs one per session, changing only `SPLIT_MODE` and `FOLD`:
   `block/0, block/1, block/2, random/0, random/1, random/2`.
6. Re-running the same fold after a timeout resumes automatically from `last.pt`.
7. After all six, run section 21 with `COMPARE_PREFIX="geonusaf-unet-r50"` for the capacity ablation.

If you OOM at batch 16 (unlikely — ResNet-18 at 512 should sit near 5-6 GB): set `BATCH_SIZE=8,
GRAD_ACCUM=2`. Effective batch stays 16, so the protocol is unchanged.

### Making the other four notebooks match
The regularization block is only protocol if it is actually in all five. What to change:

| notebook | change |
|---|---|
| `segformer-b0-new.ipynb` | already has EMA 0.999, LS 0.05, dropout 0.1, drop-path 0.1, WD 0.01, cosine horizon 120. **Add**: `MIN_DELTA=0.002`, the norm/bias decay split, `DETERMINISTIC=False`. |
| `tc-segformer.ipynb` | **Add**: EMA 0.999 + EMA-based validation/checkpointing, `LABEL_SMOOTHING=0.05`, `WEIGHT_DECAY` 1e-4 -> 0.01, norm/bias split, `DETERMINISTIC=False`. Keep the Lovasz ramp and clDice-from-epoch-20 as declared model-specific deviations. |
| `unetformer.ipynb` | Same additions as TC-SegFormer. Its LR 6e-4 / WD 2.5e-4 comes from the UNetFormer paper — keep it and flag it in the paper as an intentional outlier, but bring EMA / LS / MIN_DELTA / determinism into line. |
| `segnext-t.ipynb` | Built from the same blueprint, so paste the `# ---- REGULARIZATION` block in directly. |

Two known asymmetries to resolve before the table is publishable, both from your own notes:
- **Cosine horizon.** SegFormer-B0 ran ~160 epochs against a 120-epoch horizon; TC-SegFormer ran a 120-epoch
  horizon with `MAX_EPOCHS=120`. Those are different schedules. `MAX_EPOCHS=200` + `COS_HORIZON_EPOCHS=120`
  is the version to standardize on.
- **Determinism.** Every baseline so far paid the ~6x D6 penalty. If you re-run any of them without it, note
  in the paper that wall-clock and FPS columns are not comparable across the two groups — or re-measure the
  efficiency block for all five under the same setting, which is cheap since it is inference-only.

### What to write in the paper
Two sentences worth having ready:

> The U-Net baseline uses a ResNet-18 encoder rather than the more common ResNet-50 so that its encoder is
> identical to UNetFormer's, making the two directly comparable as a CNN decoder versus a global-local
> transformer decoder under matched capacity. A ResNet-50 variant (32.5 M parameters) is reported alongside
> it as a capacity ablation.

> All models share an identical regularization protocol (AdamW, weight decay 0.01 with normalization and
> bias parameters exempt, label smoothing 0.05, dropout 0.1, EMA 0.999 used for validation and model
> selection, 500-step warmup with cosine annealing over a 120-epoch horizon, early stopping on validation
> mIoU with patience 25 and a 0.002 minimum-improvement threshold); only the architecture and its learning
> rate differ between rows.